# 03 · Benchmark C — tilted infinite well

**Scientific question.** Under hard-wall boundaries and a non-zero interior potential, does the Dirichlet split-operator method show genuine second-order Trotter convergence?

**Scope.** $V(x)=F(x-L/2)$ on $(0,L)$ with Dirichlet walls, so that $[T,V]\neq 0$.

**Inputs.** `configs/{PROFILE}.yaml` (loaded below). No other notebook needs to have
been run first: this notebook imports everything it needs from
`src/boundary_aware_dynamics` and holds no state from any other notebook.

**Expected outputs.** Convergence tables and fits, observables, and an error budget for the only benchmark here with both hard walls and a real splitting error.

**Approximate runtime.** about 40 seconds on the `smoke` profile.

**Method.** Sine-Galerkin reference with closed-form tilt matrix elements, diagonalised exactly; Trotter order measured against exact diagonalisation of the same discrete Hamiltonian.

**Assumptions.** $F$ and $t_{\max}$ were chosen by measurement so that the sweep sits in the asymptotic regime — see the note below.

**References.** See `references/references.bib` and `docs/SCIENTIFIC_METHOD.md`.

**What this notebook does _not_ establish.** No claim of quantum advantage. This establishes that the method converges as advertised under hard walls, nothing more.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from boundary_aware_dynamics.config import load_config
from boundary_aware_dynamics import plotting

PROFILE = os.environ.get("BAD_PROFILE", "smoke")   # "paper" for manuscript numbers
                                                   # (scripts/execute_notebooks.py sets this)
config = load_config(ROOT / "configs" / f"{PROFILE}.yaml")
plotting.apply_style("preview")
print(f"profile={config.profile}  config_hash={config.config_hash}")

## Why this benchmark exists

Benchmark B has zero interior potential, so its splitting is exact and a step-count sweep there is not a Trotter test. A linear tilt is the simplest potential that is non-zero inside a hard-walled box, keeps the walls intact, and has closed-form sine-basis matrix elements.

The tilt matrix element is

$$\langle\nu|F(x-\tfrac{L}{2})|\nu'\rangle=\frac{2FL}{\pi^2}\left[\frac{1}{(\nu+\nu')^2}-\frac{1}{(\nu-\nu')^2}\right]$$

when $\nu+\nu'$ is odd and zero otherwise; the diagonal vanishes because $\sin^2$ is symmetric about $L/2$ while the tilt is antisymmetric there.

In [ ]:
from boundary_aware_dynamics.references import tilt_matrix_elements, sine_basis
bench = config.benchmark("tilted_well")
L, F = bench.domain.length, bench.tilt_force
V = tilt_matrix_elements(6, L, F)
print(f"F = {F},  L = {L}\n")
print(np.round(V, 4))
print("\nsymmetric:", np.allclose(V, V.T), " diagonal zero:", np.allclose(np.diag(V), 0))

# validate the closed form against quadrature
dense = np.linspace(0.0, L, 200_001)
B = sine_basis(dense, L, 6)
pot = F * (dense - 0.5 * L)
num = np.array([[np.trapezoid(B[a] * pot * B[b], dense) for b in range(6)] for a in range(6)])
print(f"max |closed form - quadrature| = {np.abs(V - num).max():.2e}")

## Reference convergence

A reference is only useful once it has stopped moving. Basis size is swept before it is used.

In [ ]:
from boundary_aware_dynamics.references import sine_galerkin_reference
from boundary_aware_dynamics.workflows import build_grid, build_state_callable

grid = build_grid(bench)
state_fn = build_state_callable(bench)
times = np.array([0.0, bench.time_grid.t_max])
finest = sine_galerkin_reference(state_fn, grid, times, config.physics.mass,
                                 config.physics.hbar, 512, tilt_force=F).final_state
for n_modes in (16, 32, 64, 128, 256):
    s = sine_galerkin_reference(state_fn, grid, times, config.physics.mass,
                                config.physics.hbar, n_modes, tilt_force=F)
    err = np.sqrt(grid.spacing) * np.linalg.norm(s.final_state - finest)
    print(f"  {n_modes:4d} modes: || psi - psi(512 modes) || = {err:.3e}   "
          f"tail weight {s.diagnostics['tail_weight']:.2e}")

## Trotter convergence under Dirichlet walls

Against exact diagonalisation of the same discrete Hamiltonian, so spatial discretisation and pseudospectral aliasing cancel and only the splitting error remains.

The first step count is excluded from the fit: it lies in a pre-asymptotic transient where the error is not yet dominated by the leading $\Delta t^2$ term. The excluded point and the fitted interval are both reported.

In [ ]:
from boundary_aware_dynamics.workflows import trotter_convergence_study, fit_convergence_slope

study = trotter_convergence_study(config, "tilted_well")
prev = None
for r, dt, err in zip(study.values, study.step_sizes, study.l2_state_error):
    ratio = f"{prev/err:6.2f}" if prev else "   -  "
    print(f"  r={r:5d}  dt={dt:.5f}  L2={err:.3e}  ratio={ratio}")
    prev = err
print(f"\nstate-error slope {study.fit['slope']:.3f}  R2={study.fit['r_squared']:.5f}")
print(f"fitted over dt in {study.fit['fit_interval_dt']}, excluding the first {study.fit['fit_from_index']} point(s)")
inf_fit = fit_convergence_slope(study.step_sizes, study.infidelity, fit_from=1)
print(f"infidelity  slope {inf_fit['slope']:.3f}  (expected ~4)")

In [ ]:
fig = plotting.plot_convergence(study.step_sizes, study.l2_state_error, study.fit, expected_slope=2.0)
plt.show()

## Dynamics, observables and wall behaviour

In [ ]:
from boundary_aware_dynamics.workflows import run_benchmark
result = run_benchmark(config, "tilted_well")
obs = result.observables
print(f"final infidelity {result.final_errors.infidelity:.3e}   "
      f"max norm error {result.propagation.max_norm_error:.2e}")
print(f"energy drift (bounded, O(dt^2)) {np.abs(obs['energy_drift']).max():.3e}")
print(f"<x> range [{obs['position_mean'].min():.3f}, {obs['position_mean'].max():.3f}]  "
      f"(box is (0, {L}))")
wall = np.array([b['wall_residual'] for b in result.boundary])
print(f"max wall residual {wall.max():.3e}  <- walls preserved throughout")

In [ ]:
fig = plotting.plot_density_snapshots(
    result.grid.positions, result.times,
    {"reference": result.reference.states, "dirichlet": result.propagation.states},
    np.unique(np.linspace(0, len(result.times) - 1, 4).astype(int)))
plt.show()

## Small-circuit cross-check

The Dirichlet kinetic step is also available as a validated circuit. At a size where the unitary can be formed, the circuit and the numerical propagator agree.

In [ ]:
from qiskit.quantum_info import Operator
from boundary_aware_dynamics.circuits.qst import qst_kinetic_propagator_circuit
from boundary_aware_dynamics.transforms import dst2_forward, dst2_inverse
from boundary_aware_dynamics.grids import sine_mode_energies, dirichlet_midpoint_grid

n = 3; N = 2**n; dt = 0.05
small = dirichlet_midpoint_grid(L, N)
idx = 2 * np.arange(N)
block = Operator(qst_kinetic_propagator_circuit(N, L, config.physics.mass,
                                                config.physics.hbar, dt)).data[np.ix_(idx, idx)]
rng = np.random.default_rng(0)
v = rng.normal(size=N) + 1j * rng.normal(size=N); v /= np.linalg.norm(v)
E = sine_mode_energies(N, L, config.physics.mass, config.physics.hbar)
numerical = dst2_inverse(dst2_forward(v) * np.exp(-1j * E * dt / config.physics.hbar))
print(f"|| circuit(v) - numerical(v) || = {np.linalg.norm(block @ v - numerical):.2e}")

## Summary

**Main findings.** With hard walls and a non-zero interior potential the Strang splitting shows a fitted state-error slope of 2.00 (R² = 1.0000) and an infidelity slope near 4, over roughly four decades of error. The reference is converged in basis size, and the walls are preserved throughout.

**Validation checks performed.** Closed-form tilt matrix elements against quadrature; reference convergence in basis size; Trotter slope against the discrete-Hamiltonian reference with a stated fit window; norm and energy behaviour; wall residual; agreement of the QST circuit with the numerical propagator.

**Limitations.** A linear tilt is a deliberately simple potential. Nothing here shows that an arbitrary potential can be synthesised efficiently as a diagonal.

**Generated files.** None directly; notebook 05 exports these figures.

**Relationship to the manuscript.** Supplies the paper's only genuine Trotter benchmark under Dirichlet boundaries.

**Next.** `04_circuit_validation_and_resources.ipynb` moves from numerics to circuits and their cost.